# Setting Up

In [3]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import os
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [4]:
!nvidia-smi

Sat Aug  8 10:47:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.88                 KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P0             27W /  175W |       0MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# The Optimizer & `requires_grad`

Before we can train a network, we need an engine to update its weights. In PyTorch, this is called an **Optimizer** (e.g., SGD, Adam). 

The optimizer needs to know *exactly* which tensors it is allowed to modify. It figures this out via the `requires_grad` attribute.
* If `requires_grad=True`: PyTorch tracks every mathematical operation performed on this tensor to calculate its gradient later. The optimizer will update it.
* If `requires_grad=False`: The tensor is completely ignored by the Autograd engine. This saves massive amounts of VRAM.

Sometimes we want to "freeze" the early layers of a network so they don't change, and only train the final output head. We do this by manually switching `requires_grad` to `False` for specific layers before handing them to the optimizer.

In [5]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)
for name, param in model.named_parameters():
    print(f"Layer: {name} | Requires Grad: {param.requires_grad}")

Layer: 0.weight | Requires Grad: True
Layer: 0.bias | Requires Grad: True
Layer: 2.weight | Requires Grad: True
Layer: 2.bias | Requires Grad: True


In [6]:
model[0].weight.requires_grad = False
model[0].bias.requires_grad = False
for name, param in model.named_parameters():
    print(f"Layer: {name:<15} | Requires Grad: {param.requires_grad}")

Layer: 0.weight        | Requires Grad: False
Layer: 0.bias          | Requires Grad: False
Layer: 2.weight        | Requires Grad: True
Layer: 2.bias          | Requires Grad: True


In [7]:
# Don't just pass model.parameters(). 
# Explicitly filter and only pass the parameters that actually require gradients.
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = optim.Adam(trainable_params, lr=0.001)
print(f"{len(trainable_params)} parameter matrices")

2 parameter matrices


### Sometimes we want the first layers to learn, but very slowly

In [8]:
model[0].weight.requires_grad = True
model[0].bias.requires_grad = True

# We pass a list of dictionaries instead of a single list of parameters.
advanced_optimizer = optim.Adam([
    # Group 0: Feature Extractor (Learns 100x slower to preserve features)
    {"params": model[0].parameters(), "lr": 1e-5}, 
    
    # Group 1: Output Head (Learns at a normal speed)
    {"params": model[2].parameters(), "lr": 1e-3}  
], weight_decay=1e-4) # Global L2 Regularization applied to all groups

# ==========================================
# 3. Inspecting the Optimizer's Memory
# ==========================================
for i, group in enumerate(advanced_optimizer.param_groups):
    num_tensors = len(group['params'])
    learning_rate = group['lr']
    weight_decay = group['weight_decay']
    
    print(f"Group {i} | Tensors Managed: {num_tensors} | LR: {learning_rate} | Decay: {weight_decay}")

Group 0 | Tensors Managed: 2 | LR: 1e-05 | Decay: 0.0001
Group 1 | Tensors Managed: 2 | LR: 0.001 | Decay: 0.0001


### Important parameters for optimizers

- decoupled_weight_decay: This is very important for SOTA Transformers. Standard Adam applies weight decay in a mathematically flawed way (it mixes it into the gradient momentum). Setting this to True applies the decay purely to the weights themselves.

- maximize: By default, optimizers do Gradient Descent (they minimize the Loss). Setting this to True flips the math to do Gradient Ascent.

- foreach:  If True, PyTorch updates the weights using a highly optimized, vectorized C++ loop rather than a slower, element-by-element Python for loop. It is usually set to None so PyTorch can automatically figure out if your hardware supports it.

- fused: If True, PyTorch bundles all the optimizer math into a single, massive GPU kernel. Instead of the GPU reading/writing memory 5 separate times per step, it does it all in one lightning-fast operation. Highly recommended for massive models if you have a modern NVIDIA GPU.

- differentiable: By default, you can't run backpropagation through the optimizer itself. If True, it forces Autograd to track the optimizer's math too. It is used heavily in Meta-Learning (teaching a network how to learn, so you need the derivative of the learning process itself).



# The 5-Step Core Loop



Training a neural network is an optimization cycle. We pass data through, measure our error, calculate the slope of that error (gradients), and nudge the weights down the slope.

1. Forward Pass -> `model(x)`
2. Compute Loss -> `criterion(prediction, target)`
3. Zero Gradients -> `optimizer.zero_grad()`
4. Backpropagate -> `loss.backward()`
5. Update Weights -> `optimizer.step()`

If you forget step 3 (`zero_grad`), PyTorch will simply add the new gradients on top of the old ones. Your network will take massive, wildly incorrect steps and crash into `NaN` (infinity).

In [9]:
model = nn.Linear(10, 2) 

# Dummy Data (Batch Size of 32)
data = torch.randn(32, 10)
# Dummy Targets (32 random integers: 0 or 1)
targets = torch.randint(0, 2, (32,)) 

# The standard loss function for classification
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.01)

In [10]:
for epoch in range(3):
    
    # STEP 1: Forward Pass (Predict)
    predictions = model(data)
    
    # STEP 2: Compute Loss (Measure Error)
    loss = criterion(predictions, targets)
    
    # STEP 3: Zero Gradients (Clear the Tape Recorder)
    optimizer.zero_grad()
    
    # STEP 4: Backward Pass (Calculate Gradients)
    loss.backward()
    
    # STEP 5: Optimizer Step (Update Weights)
    optimizer.step()
    
    print(f"Epoch {epoch} | Loss: {loss.item()}")

Epoch 0 | Loss: 0.7140188813209534
Epoch 1 | Loss: 0.7020708918571472
Epoch 2 | Loss: 0.6908794641494751


### The Production-Ready Loop


To make this script robust enough,  we must add two things:
1. `model.train()`: Ensures layers like Dropout and BatchNorm are behaving correctly.
2. `clip_grad_norm_`: Acts as a strict speed limit on our gradients to prevent mathematical explosions

In [11]:
model.train() # use model.eval() for evaluation
for epoch in range(3):
    
    # 1. Forward Pass
    predictions = model(data)
    
    # 2. Compute Loss
    loss = criterion(predictions, targets)
    
    # 3. Zero Gradients
    optimizer.zero_grad()
    
    # 4. Backward Pass (Calculate the raw gradients)
    loss.backward()
    
    # If any gradient vector exceeds a length of 1.0, scale it down proportionally.
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # 5. Optimizer Step (Apply the safely clipped gradients)
    optimizer.step()
    
    print(f"Epoch {epoch} | Loss: {loss.item()}")

Epoch 0 | Loss: 0.6802973747253418
Epoch 1 | Loss: 0.67032390832901
Epoch 2 | Loss: 0.6609792709350586


# Model Evaluation & Inference

When a mdoel stops training and plays a test game to see how good it got, we must switch the framework into Evaluation Mode. This requires two mandatory commands that do completely different things:

1. **`model.eval()` (The Architecture Switch):** 
   This tells layers like Dropout and BatchNorm to stop modifying the data. If you don't use this, your network will randomly drop neurons during your test game, causing erratic, terrible performance.
   
2. **`with torch.no_grad():` (The Memory Switch):** 
   `model.eval()` does *not* turn off the Autograd tape recorder. If you run a test loop without `no_grad()`, PyTorch will still build a massive computational graph in the background, eventually maxing out your VRAM and crashing your GPU.

In [12]:
test_data = torch.randn(10, 10)
test_targets = torch.randint(0, 2, (10,))
model.eval()

total_test_loss = 0.0

# 2. Flip the Engine Switch (Disable the tape recorder)
with torch.no_grad():
    
    # Everything inside this block uses ~70% less memory and runs much faster
    predictions = model(test_data)
    loss = criterion(predictions, test_targets)
    
    total_test_loss += loss.item()
    
print(f"Test Loss: {total_test_loss}")

Test Loss: 0.8068803548812866


While `torch.no_grad()` is the classic way to evaluate models, PyTorch recently introduced `torch.inference_mode()`. 

It is a strictly faster, highly optimized context manager that completely disables view tracking and version counters in the C++ backend. If you are doing pure inference (evaluating an agent or deploying to production), this is the modern standard.

In [13]:
with torch.inference_mode():
    
    # This runs at the absolute maximum speed your hardware allows
    fast_predictions = model(test_data)

# Checkpointing (Saving & Loading)

If your laptop dies during epoch 99 of a 100-epoch training run, you do not want to start over. To resume training, you must save both the **Model's state** (the weights) AND the **Optimizer's state** (its momentum and learning rate history). 

**The Best Practice:** We bundle the model's `state_dict`, the optimizer's `state_dict`, and the current `epoch` into a standard Python dictionary and save *that* dictionary to the hard drive. This is called a Checkpoint.

In [14]:
checkpoint_path = "checkpoint.pth"
# We create a dictionary containing everything needed to resume training exactly where we left off
checkpoint = {
    'epoch': 3,  # The epoch we just finished
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.432  # Good for keeping track of performance
}

torch.save(checkpoint, checkpoint_path)
print(f"File Size: {os.path.getsize(checkpoint_path) / 1024:.2f} KB\n")



# Imagine we just restarted our computer. We must first instantiate fresh, untrained objects:
fresh_model = nn.Linear(10, 2)
fresh_optimizer = optim.Adam(fresh_model.parameters(), lr=0.01)

# Read the file from the hard drive
# (Note: weights_only=True is a modern PyTorch security best practice to prevent loading malicious code)
loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)

# Physically inject the saved memories into the fresh objects
fresh_model.load_state_dict(loaded_checkpoint['model_state_dict'])
fresh_optimizer.load_state_dict(loaded_checkpoint['optimizer_state_dict'])
resumed_epoch = loaded_checkpoint['epoch']

print(f"Model and Optimizer states successfully injected")
print(f"Ready to resume training from Epoch {resumed_epoch + 1}.")

File Size: 3.93 KB

Model and Optimizer states successfully injected
Ready to resume training from Epoch 4.


# Customs

We usually have to translate complex math formulas (like the Bellman equation or Policy Gradients) directly into code.

A loss function in PyTorch is simply a standard Python function (or an `nn.Module`) that takes in Tensors, performs math on them, and returns a single scalar Tensor. 

As long as you **only use PyTorch operations** (e.g., `torch.mean()`, `torch.sum()`, `torch.abs()`), the Autograd tape recorder will automatically track your custom math, and `loss.backward()` will work flawlessly without you ever doing the calculus!

In [15]:
class MyCustomRLLoss(nn.Module):
    def __init__(self, penalty_weight=0.1):
        super().__init__()
        self.penalty_weight = penalty_weight

    def forward(self, predictions, targets):
        """
        Imagine this is a custom RL scenario:
        We want standard Mean Squared Error, BUT we want to heavily 
        penalize the agent if its prediction goes above 1.0 (maybe going 
        above 1.0 causes a robot arm to crash into a wall).
        """
        
        mse = torch.mean((predictions - targets) ** 2)
        
        # 2. Custom Penalty Math: Only penalize values > 1.0
        # torch.relu returns 0 for negative numbers, keeping only the excess
        overshoot_penalty = torch.mean(torch.relu(predictions - 1.0))
        
        # 3. Combine them into a single scalar tensor
        total_loss = mse + (self.penalty_weight * overshoot_penalty)
        
        return total_loss

In [18]:
dummy_predictions = torch.tensor([0.5, 1.2, 0.9, 1.5], requires_grad=True)
dummy_targets = torch.tensor([0.5, 1.0, 1.0, 1.0])

custom_criterion = MyCustomRLLoss(penalty_weight=0.5)

loss = custom_criterion(dummy_predictions, dummy_targets)
print(f"Calculated Custom Loss: {loss.item()}")

loss.backward()

# Inspect the Gradients
print(f"Gradients on Predictions: {dummy_predictions.grad}")

Calculated Custom Loss: 0.16250000894069672
Gradients on Predictions: tensor([ 0.0000,  0.2250, -0.0500,  0.3750])
